# 04 – E1: Baseline Truyền Thống (Canny + Dense HOG + BoVW + Spatial Pyramid) (Thành viên A)

Thực hiện theo **Mục 2.4 & Phụ lục A** trong Kế hoạch thực nghiệm 4 tuần (v2):
- **Ảnh thật (Photo)**: Đọc từ cache cạnh ngắn 256px (`sbir-cache-256-photos`). Tiền xử lý: bilateral filter -> Canny ngưỡng động -> lọc Connected Components biên rác -> crop bbox có biên cạnh (10% padding) -> resize 256x256 -> trích xuất Dense HOG.
- **Nét vẽ (Sketch)**: Đọc trực tiếp từ raw (`sketchyextended`, `flickr15k`). Tiền xử lý: Otsu -> crop bbox nét vẽ (10% padding) -> làm mảnh Zhang-Suen -> giãn nhẹ 3x3 -> resize 256x256 -> trích xuất Dense HOG.
- **Visual Codebook (K=1000)**: MiniBatchKMeans học từ 300.000 descriptors lấy mẫu từ tập train (94 lớp seen của Sketchy).
- **Spatial Pyramid Matching (SPM 5000-D)**: Level 1 (1x1) + Level 2 (2x2) = 5 ô không gian x 1000 từ = 5000 chiều. Có áp dụng trọng số smooth TF-IDF, chuẩn hóa Power-norm (signed square-root) và chuẩn hóa L2.
- **Đánh giá & Bàn giao**: Cosine similarity trên Sketchy Test (21 unseen classes) và Flickr15K (33 classes). Xuất `results/E1/metrics.json` và lưu embeddings chuẩn bàn giao `features/E1/<dataset>_<split>_<sketch|photo>.npy|csv`.

Chạy trên **Kaggle CPU** (hoặc GPU), **Internet = On**.

In [ ]:
# ======= CẤU HÌNH ĐƯỜNG DẪN =======
import os, glob

# 1. Đường dẫn ảnh thật đã cache 256px
CANDIDATE_CACHE = [
    "/kaggle/input/sbir-cache-256-photos",
    "/kaggle/input/datasets/sbir-cache-256-photos",
    *glob.glob("/kaggle/input/*sbir-cache-256-photos*"),
]
CACHE_PHOTO_ROOT = next((p for p in CANDIDATE_CACHE if os.path.exists(p)), "/kaggle/input/sbir-cache-256-photos")

# 2. Đường dẫn Sketchy gốc (đọc sketch)
CANDIDATE_SKETCHY = [
    "/kaggle/input/datasets/hongonthi/sketchyextended/sketchy",
    "/kaggle/input/sketchyextended/sketchy",
    *glob.glob("/kaggle/input/**/sketchy extended/sketchy", recursive=True),
    *glob.glob("/kaggle/input/**/sketchy", recursive=True),
]
SKETCHY_RAW = next((p for p in CANDIDATE_SKETCHY if os.path.exists(p) and os.path.isdir(p)), "/kaggle/input/datasets/hongonthi/sketchyextended/sketchy")

# 3. Đường dẫn Flickr15K gốc (đọc sketch)
CANDIDATE_FLICKR = [
    "/kaggle/input/datasets/hongonthi/flickr15k/Flickr15K",
    "/kaggle/input/flickr15k/Flickr15K",
    *glob.glob("/kaggle/input/**/Flickr15K", recursive=True),
]
FLICKR_RAW = next((p for p in CANDIDATE_FLICKR if os.path.exists(p) and os.path.isdir(p)), "/kaggle/input/datasets/hongonthi/flickr15k/Flickr15K")

REPO = "https://github.com/thaihoang104569/CS406.git"

print(f"Cache Photo Root: {CACHE_PHOTO_ROOT} (exists: {os.path.exists(CACHE_PHOTO_ROOT)})")
print(f"Sketchy Raw:      {SKETCHY_RAW} (exists: {os.path.exists(SKETCHY_RAW)})")
print(f"Flickr15K Raw:    {FLICKR_RAW} (exists: {os.path.exists(FLICKR_RAW)})")

In [ ]:
# ======= CLONE HOẶC PULL REPO =======
WORK = "/kaggle/working/CS406"
if os.path.isdir(WORK):
    !git -C {WORK} pull
else:
    !git clone {REPO} {WORK}

!git -C {WORK} log --oneline -2

In [ ]:
# Kiểm tra thư viện
!pip install -q scikit-learn opencv-python pillow pandas numpy tqdm
import cv2, sklearn, pandas, numpy
print(f"OpenCV version: {cv2.__version__}")
print(f"Scikit-Learn:  {sklearn.__version__}")

## 1. Chuẩn bị tập dữ liệu phân chia (Dataset Splits)
Tạo các file split CSV cố định (`sketchy_zero_shot_splits.csv` và `flickr15k_splits.csv`) bằng `prepare_data.py` bám sát 10 lớp val đã định trước.

In [ ]:
!python {WORK}/scripts/prepare_data.py \
    --sketchy_root "{SKETCHY_RAW}" \
    --flickr_root "{FLICKR_RAW}" \
    --out_dir /kaggle/working/data_prep \
    --class_split_csv {WORK}/data/splits/sketchy_class_splits.csv \
    --skip_dup_check \
    --no_figures \
    --sample_per_group 50

## 2. Kiểm tra nhanh (Debug Run, ~1-2 phút)
Chạy thử với một lượng nhỏ mẫu (`--debug 30` mỗi lớp) để xác nhận toàn bộ quy trình: trích xuất Dense HOG -> gom cụm MiniBatchKMeans -> sinh SPM 5000-D -> tính Cosine Similarity -> tính mAP@200.

In [ ]:
SPLITS_DIR = "/kaggle/working/data_prep/splits"
SK_PHOTO_DIR = f"{CACHE_PHOTO_ROOT}/sketchy"
FL_PHOTO_DIR = f"{CACHE_PHOTO_ROOT}/flickr15k"

# Chạy debug nhanh
!python {WORK}/scripts/run_e1_baseline.py \
    --sketchy_csv {SPLITS_DIR}/sketchy_zero_shot_splits.csv \
    --sketchy_photo_root "{SK_PHOTO_DIR}" \
    --sketchy_sketch_root "{SKETCHY_RAW}" \
    --flickr_csv {SPLITS_DIR}/flickr15k_splits.csv \
    --flickr_photo_root "{FL_PHOTO_DIR}" \
    --flickr_sketch_root "{FLICKR_RAW}" \
    --out_dir /kaggle/working/debug_e1/results \
    --feat_dir /kaggle/working/debug_e1/features \
    --weights_dir /kaggle/working/debug_e1/weights \
    --k_words 200 \
    --sample_descriptors 50000 \
    --workers 4 \
    --debug 30

## 3. Thực thi đầy đủ E1 Baseline (Full Execution)
- Codebook K=1000 từ 300.000 descriptors tập train seen.
- Trích xuất và đánh giá trên Sketchy Test (21 unseen classes) và Flickr15K (33 classes).
- Toàn bộ embedding SPM 5000-D và file ID được xuất ra thư mục `/kaggle/working/features/E1/`.

In [ ]:
!python {WORK}/scripts/run_e1_baseline.py \
    --sketchy_csv {SPLITS_DIR}/sketchy_zero_shot_splits.csv \
    --sketchy_photo_root "{SK_PHOTO_DIR}" \
    --sketchy_sketch_root "{SKETCHY_RAW}" \
    --flickr_csv {SPLITS_DIR}/flickr15k_splits.csv \
    --flickr_photo_root "{FL_PHOTO_DIR}" \
    --flickr_sketch_root "{FLICKR_RAW}" \
    --out_dir /kaggle/working/results/E1 \
    --feat_dir /kaggle/working/features/E1 \
    --weights_dir /kaggle/working/weights/E1 \
    --k_words 1000 \
    --sample_descriptors 300000 \
    --workers 4

## 4. Báo cáo kết quả và Đóng gói bàn giao
Hiển thị bảng chỉ số mAP, mAP@200, P@10, P@100, P@200 và nén kết quả nộp đồ án.

In [ ]:
import json, shutil
import pandas as pd
from IPython.display import display

metrics_path = "/kaggle/working/results/E1/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path, "r", encoding="utf-8") as f:
        m = json.load(f)

    print(f"Experiment: {m.get('experiment')} | Commit: {m.get('git_commit')}")
    summary_rows = []
    for dataset_key, data in m.get("results", {}).items():
        v = data.get("visual", {})
        summary_rows.append({
            "Dataset": dataset_key,
            "Queries (Sketches)": data.get("n_sketches"),
            "Gallery (Photos)": data.get("n_photos"),
            "Classes": data.get("n_classes"),
            "mAP": round(v.get("mAP", 0.0), 4),
            "mAP@200": round(v.get("mAP@200", 0.0), 4),
            "mAP@200_topk_hit": round(v.get("mAP@200_topk_hit", 0.0), 4),
            "P@10": round(v.get("P@10", 0.0), 4),
            "P@100": round(v.get("P@100", 0.0), 4),
            "P@200": round(v.get("P@200", 0.0), 4),
            "Sim Time (s)": data.get("sim_time_sec"),
        })

    df_summary = pd.DataFrame(summary_rows).set_index("Dataset")
    print("\n--- TỔNG HỢP CHỈ SỐ E1 TRADITIONAL BASELINE ---")
    display(df_summary)

    if "sketchy_test" in m.get("results", {}):
        pc = pd.DataFrame(m["results"]["sketchy_test"].get("per_class", {})).T
        if not pc.empty:
            pc = pc.sort_values("mAP@200", ascending=False)
            print("\n--- CHI TIẾT THEO LỚP (Sketchy Test 21 Unseen Classes) ---")
            display(pc)

    # Nén artifact để tải về
    print("\nĐang đóng gói E1_outputs.zip và E1_features.zip...")
    shutil.make_archive("/kaggle/working/E1_outputs", "zip", "/kaggle/working", "results/E1")
    shutil.make_archive("/kaggle/working/E1_features", "zip", "/kaggle/working", "features/E1")
    shutil.make_archive("/kaggle/working/E1_weights", "zip", "/kaggle/working", "weights/E1")
    !ls -lh /kaggle/working/*.zip
    print("\n✅ Hoàn thành đóng gói! Có thể tải các file zip từ mục Output của Kaggle.")
else:
    print(f"Chưa tìm thấy {metrics_path}. Vui lòng kiểm tra lại quá trình chạy.")